# Neural Networks

## 1. linear的问题

- geometric （圆环）
- visual （双头马）

## 2. 解决方法

### 2.1 Feature Transformation: Hand Crafted feature: SIFT, HoG

手工设计特征

### 2.2 Feature Transformation: Bag of Words

从raw中聚类为词典作为特征，也算是data driven 的方法。

### 2.3 Neural Networks

端到端的data driven，只不过可以全流程端到端优化调参，而feature transformation 只能调最后一阶段训练的参数。

## 3. Neural Networks / MLP / 全连接

### 3.1 结构

input layer -> hidden layer -> output layer

![neural network](neural_networks.png)

neural network 的层数就是 matrix 的数量。

### 3.2 解决 linear 的问题

双头马可以通过隐藏层很好地被分类：subset 可在 hidden layer 中处理。

但是会有 redundancy。

且是non-convex optimization problem。

### 3.3 原理

#### 3.3.0 key: 激活函数

why activation function?

$$
S = W_2 W_1 x
$$

仍然线性，依旧不能解决非线性逼近。

#### 3.3.1 space warping

![space warping](space_warping.png)

在这里以一个二元输入模型为例，首先第一层通过 feature transform 映射到笛卡尔空间。

![space warping2](space_warping3.png)

接着第二层根据 ReLU 激活函数，将映射到笛卡尔空间中的数据映射到第二层输出空间。

第二层输入作为第三层输入，再次 linear，可以看见数据集被很好地分开了。

![space warping3](space_warping4.png)

hidden layer 的层数越多，模型越复杂，越容易过拟合。但是我们调参数的时候不会选择调层数（太麻烦），而是用别的参数来调整拟合情况。

#### 3.3.2 universal approximation

下面用一个例子说明，采用单层 ReLU 逼近 $f: \mathbb{R} \rightarrow \mathbb{R}$ 的任意函数 $f$。

![universal approximation](universal_approximation.png)

> 两层 ReLU 神经网络可以拼出很多“小凸起 bump”，再把很多 bump 加起来，就能近似任意一维函数。

```text
0 → 上升 → 平台 → 下降 → 0
```

它用四个 ReLU 拼出来：

$$
f(x)
=
m_1\max(0,x-s_1)
-m_1\max(0,x-s_2)
-m_2\max(0,x-s_3)
+m_2\max(0,x-s_4)
$$


这四项分别负责：

| 项 | 作用 |
|---|---|
| $m_1\max(0,x-s_1)$ | 从 $s_1$ 开始往上升 |
| $-m_1\max(0,x-s_2)$ | 从 $s_2$ 开始抵消上升，让它变平 |
| $-m_2\max(0,x-s_3)$ | 从 $s_3$ 开始往下降 |
| $m_2\max(0,x-s_4)$ | 从 $s_4$ 开始抵消下降，让它回到 0 |

$$
\text{ReLU}
\rightarrow
\text{斜坡}
\rightarrow
\text{多个斜坡相加}
\rightarrow
\text{bump}
\rightarrow
\text{多个 bump 相加}
\rightarrow
\text{近似任意函数}
$$




In [ ]:
import os

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset


# Toy problem: XOR 是一个经典的线性不可分问题。
# 输入有两个 feature，输出有两个类别。
torch.manual_seed(0)

X_base = torch.tensor([
    [0.0, 0.0],
    [0.0, 1.0],
    [1.0, 0.0],
    [1.0, 1.0],
])
y_base = torch.tensor([0, 1, 1, 0])

# 重复基础样本，并加入一点噪声，让训练过程更像真实数据。
num_repeats = 100
X = X_base.repeat(num_repeats, 1) + 0.05 * torch.randn(num_repeats * 4, 2)
y = y_base.repeat(num_repeats)

dataset = TensorDataset(X, y)
loader = DataLoader(dataset, batch_size=32, shuffle=True)


# MLP: input layer -> hidden layer -> output layer
# ReLU 是关键：如果没有非线性激活函数，多层线性层叠加后仍然只是线性变换。
model = nn.Sequential(
    nn.Linear(2, 8),
    nn.ReLU(),
    nn.Linear(8, 2),
)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)


num_epochs = 200

for epoch in range(num_epochs):
    total_loss = 0.0

    for X_batch, y_batch in loader:
        logits = model(X_batch)
        loss = criterion(logits, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(X_batch)

    if epoch == 0 or (epoch + 1) % 50 == 0:
        with torch.no_grad():
            preds = model(X).argmax(dim=1)
            accuracy = (preds == y).float().mean().item()

        print(f"epoch {epoch + 1:03d}: loss={total_loss / len(dataset):.6f}, acc={accuracy:.3f}")


# 看看模型对四个原始 XOR 点的预测。
with torch.no_grad():
    logits = model(X_base)
    probs = torch.softmax(logits, dim=1)
    preds = probs.argmax(dim=1)

print("\nXOR predictions:")
for x, pred, prob in zip(X_base, preds, probs):
    print(f"x={x.tolist()}, pred={pred.item()}, prob={prob.tolist()}")
